# 命题切分（Proposition Chunking）

目标：把文档先切成常规 chunks，再用 LLM 将每个 chunk 拆成更细粒度的“命题（propositions）”。命题应当是**原子、事实、自洽、简洁**的陈述，适合做更精确的向量检索。


## 0) 导入依赖并加载环境变量


In [1]:
from __future__ import annotations

from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv("../.env")

from langchain_core.documents import Document
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_3118837/3935921976.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) 准备测试文档

这一节沿用原版做法：用一段较长的英文文本作为“待检索文档”。


In [2]:
sample_content = """Paul Graham's essay \"Founder Mode,\" published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. \"Founder Mode\" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional \"manager mode\" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graham suggests that founders should leverage these strengths rather than conform to traditional managerial practices. \"Founder Mode\" is an emerging paradigm that is not yet fully understood or documented, with Graham hoping that over time, it will become as well-understood as the traditional manager mode, allowing founders to maintain their unique approach even as their companies scale.
Challenges of Scaling Startups
As startups grow, there is a common belief that they must transition to a more structured managerial approach. However, many founders have found this transition problematic, as it often leads to a loss of the innovative and agile spirit that drove the startup's initial success.
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's \"Founder Mode\" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart.
This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale.
"""

docs_list = [
    Document(
        page_content=sample_content,
        metadata={"title": "Founder Mode Essay"},
    )
]

len(docs_list), len(docs_list[0].page_content)

(1, 2648)

## 2) 常规 Chunking

先把文档切成常规 chunk，并为每个 chunk 标注 `chunk_id`，后面生成命题/检索时会用到。


In [3]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200,
    chunk_overlap=50,
)

doc_splits = text_splitter.split_documents(docs_list)

for i, doc in enumerate(doc_splits, start=1):
    doc.metadata["chunk_id"] = i

len(doc_splits), doc_splits[0].metadata

(3, {'title': 'Founder Mode Essay', 'chunk_id': 1})

In [4]:
doc_splits

[Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1}, page_content='Paul Graham\'s essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.\nConventional Wisdom vs. Founder Mode\nThe essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.\nThis approach, suitable for established companies, can be detrimental to startups where the founder\'s vision and direct involvement are crucial. "Founder Mode" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional "manager mode" often advised by business schools and professional managers.\nUnique Founder Abilities\nFounders possess unique insights and abilities that professional managers d

In [5]:
# 看看切分后的前几个 chunk
for d in doc_splits[:3]:
    print("chunk_id=", d.metadata["chunk_id"], "len=", len(d.page_content))
    print(d.page_content[:200])
    print("---")

chunk_id= 1 len= 1006
Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than 
---
chunk_id= 2 len= 895
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graha
---
chunk_id= 3 len= 939
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a d
---


## 3) 生成命题（Propositions）

对每个 chunk，让模型输出一个 `propositions: List[str]` 的结构化结果。每条命题应当是：单一事实、无需额外上下文也能理解、尽量避免代词、必要时包含限定信息。


In [6]:
class GeneratePropositions(BaseModel):
    """一个 chunk 对应的一组命题。"""

    propositions: List[str] = Field(
        description="Factual, self-contained, concise propositions extracted from the input text"
    )


llm_prop = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_prop = llm_prop.with_structured_output(GeneratePropositions)

proposition_examples = [
    {
        "document": "In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.",
        "propositions": "['Neil Armstrong was an astronaut.', 'Neil Armstrong walked on the Moon in 1969.', 'Neil Armstrong was the first person to walk on the Moon.', 'Neil Armstrong walked on the Moon during the Apollo 11 mission.', 'The Apollo 11 mission occurred in 1969.']",
    }
]

example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{document}"), ("ai", "{propositions}")]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=proposition_examples,
)

system_prop = """Break down the following text into simple, self-contained propositions.

Rules:
1) Express a single fact per proposition.
2) The proposition must be understandable without extra context.
3) Prefer full entity names over pronouns.
4) Include necessary dates/qualifiers when relevant.
5) Avoid conjunctions that pack multiple facts into one sentence.
"""

prompt_prop = ChatPromptTemplate.from_messages(
    [
        ("system", system_prop),
        few_shot_prompt,
        ("human", "{document}"),
    ]
)

proposition_generator = prompt_prop | structured_prop

In [7]:
# 生成命题（可按需限制 chunk 数量以节省成本）
MAX_CHUNKS = None  # 例如设为 3 只处理前 3 个 chunk

proposition_docs: list[Document] = []

splits_to_process = doc_splits if MAX_CHUNKS is None else doc_splits[:MAX_CHUNKS]

for chunk in splits_to_process:
    resp = proposition_generator.invoke({"document": chunk.page_content})
    for p in resp.propositions:
        proposition_docs.append(
            Document(
                page_content=p,
                metadata={
                    **chunk.metadata,
                    "type": "proposition",
                },
            )
        )

len(proposition_docs), proposition_docs[0].metadata

(36, {'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'})

In [11]:
proposition_docs

[Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='Paul Graham published the essay "Founder Mode" in September 2024.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay "Founder Mode" challenges conventional wisdom about scaling startups.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay argues that founders should maintain their unique management style.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay argues against adopting traditional corporate practices as companies grow.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The traditional advice for growing companies is to hire good people and give them autonomy.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1

In [8]:
resp

GeneratePropositions(propositions=['Brian Chesky is the co-founder of Airbnb.', 'Brian Chesky was advised to run Airbnb in a traditional managerial style.', 'The traditional managerial style led to poor outcomes for Airbnb.', 'Brian Chesky found success by adopting a different approach.', "Brian Chesky's approach was influenced by Steve Jobs' management style at Apple.", 'Steve Jobs managed Apple in an unconventional way.', 'Steve Jobs held an annual retreat for the 100 most important people at Apple.', 'The annual retreat included individuals regardless of their position on the organizational chart.', "Steve Jobs' method maintained a startup-like environment at Apple.", "Steve Jobs' approach fostered innovation at Apple.", "Steve Jobs' management style encouraged direct communication across hierarchical levels.", "Founders should stay deeply involved in their companies' operations according to Steve Jobs' practices.", "The traditional notion of delegating responsibilities to professio

In [9]:
# 预览一些命题
for i, d in enumerate(proposition_docs[:10], start=1):
    print(f"{i}) {d.page_content} (chunk_id={d.metadata['chunk_id']})")


1) Paul Graham published the essay "Founder Mode" in September 2024. (chunk_id=1)
2) The essay "Founder Mode" challenges conventional wisdom about scaling startups. (chunk_id=1)
3) The essay argues that founders should maintain their unique management style. (chunk_id=1)
4) The essay argues against adopting traditional corporate practices as companies grow. (chunk_id=1)
5) The traditional advice for growing companies is to hire good people and give them autonomy. (chunk_id=1)
6) The traditional advice often fails when applied to startups. (chunk_id=1)
7) The approach suitable for established companies can be detrimental to startups. (chunk_id=1)
8) The founder's vision and direct involvement are crucial for startups. (chunk_id=1)
9) "Founder Mode" is an emerging paradigm that is not yet fully understood or documented. (chunk_id=1)
10) "Founder Mode" contrasts with the conventional "manager mode" advised by business schools. (chunk_id=1)


## 4) 命题质量检查（Quality Check）

对每条命题从 4 个维度打分（1-10）：准确性、清晰度、完整性、简洁性。低于阈值的命题丢弃。


In [10]:
class GradeProposition(BaseModel):
    accuracy: int = Field(description="1-10: how well the proposition reflects the original text")
    clarity: int = Field(description="1-10: understandable without extra context")
    completeness: int = Field(description="1-10: includes necessary qualifiers/details")
    conciseness: int = Field(description="1-10: concise without losing key information")


llm_grade = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_grade = llm_grade.with_structured_output(GradeProposition)

system_grade = """You are grading a single proposition against the original text.
Return scores as integers 1-10 for: accuracy, clarity, completeness, conciseness.
"""

prompt_grade = ChatPromptTemplate.from_messages(
    [
        ("system", system_grade),
        ("human", "Proposition:\n{proposition}\n\nOriginal Text:\n{original_text}"),
    ]
)

grader = prompt_grade | structured_grade

thresholds = {
    "accuracy": 7,
    "clarity": 7,
    "completeness": 7,
    "conciseness": 7,
}

In [12]:
def passes(scores: GradeProposition) -> bool:
    return (
        scores.accuracy >= thresholds["accuracy"]
        and scores.clarity >= thresholds["clarity"]
        and scores.completeness >= thresholds["completeness"]
        and scores.conciseness >= thresholds["conciseness"]
    )


evaluated_propositions: list[Document] = []
failed: list[tuple[Document, GradeProposition]] = []

for prop_doc in proposition_docs:
    chunk_id = prop_doc.metadata["chunk_id"]
    original_text = doc_splits[chunk_id - 1].page_content
    scores = grader.invoke({"proposition": prop_doc.page_content, "original_text": original_text})

    if passes(scores):
        evaluated_propositions.append(prop_doc)
    else:
        failed.append((prop_doc, scores))

len(evaluated_propositions), len(failed)

(30, 6)

In [13]:
# 看几个被淘汰的例子（可选）
for i, (d, s) in enumerate(failed[:5], start=1):
    print(f"{i}) chunk_id={d.metadata['chunk_id']} proposition={d.page_content}")
    print(
        f"   scores: accuracy={s.accuracy}, clarity={s.clarity}, completeness={s.completeness}, conciseness={s.conciseness}"
    )


1) chunk_id=2 proposition='Founder Mode' is an emerging paradigm.
   scores: accuracy=9, clarity=7, completeness=6, conciseness=8
2) chunk_id=3 proposition=Brian Chesky is the co-founder of Airbnb.
   scores: accuracy=10, clarity=10, completeness=5, conciseness=8
3) chunk_id=3 proposition=Brian Chesky was advised to run Airbnb in a traditional managerial style.
   scores: accuracy=6, clarity=7, completeness=5, conciseness=8
4) chunk_id=3 proposition=Steve Jobs managed Apple in an unconventional way.
   scores: accuracy=8, clarity=7, completeness=6, conciseness=9
5) chunk_id=3 proposition=Steve Jobs' method maintained a startup-like environment at Apple.
   scores: accuracy=8, clarity=7, completeness=6, conciseness=9


## 5) 为命题构建向量库并检索


In [14]:
evaluated_propositions

[Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='Paul Graham published the essay "Founder Mode" in September 2024.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay "Founder Mode" challenges conventional wisdom about scaling startups.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay argues that founders should maintain their unique management style.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The essay argues against adopting traditional corporate practices as companies grow.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1, 'type': 'proposition'}, page_content='The traditional advice for growing companies is to hire good people and give them autonomy.'),
 Document(metadata={'title': 'Founder Mode Essay', 'chunk_id': 1

In [15]:
embeddings = OpenAIEmbeddings()

vectorstore_propositions = FAISS.from_documents(evaluated_propositions, embeddings)
retriever_propositions = vectorstore_propositions.as_retriever(search_kwargs={"k": 4})

# 基于命题的检索
query = "Whose management approach served as inspiration for Brian Chesky's \"Founder Mode\" at Airbnb?"
res_proposition = retriever_propositions.invoke(query)

for i, r in enumerate(res_proposition, start=1):
    print(f"{i}) {r.page_content} (chunk_id={r.metadata['chunk_id']})")


1) Brian Chesky's approach was influenced by Steve Jobs' management style at Apple. (chunk_id=3)
2) Brian Chesky found success by adopting a different approach. (chunk_id=3)
3) The essay "Founder Mode" challenges conventional wisdom about scaling startups. (chunk_id=1)
4) "Founder Mode" contrasts with the conventional "manager mode" advised by business schools. (chunk_id=1)


## 6) 与“更大 chunk 检索”对比

用同样的 query，对比直接用常规 chunk 的向量库检索返回的内容。


In [ ]:
vectorstore_larger = FAISS.from_documents(doc_splits, embeddings)
retriever_larger = vectorstore_larger.as_retriever(search_kwargs={"k": 4})

res_larger = retriever_larger.invoke(query)

for i, r in enumerate(res_larger, start=1):
    print(f"{i}) (chunk_id={r.metadata['chunk_id']})")
    print(r.page_content)
    print("---")

1) (chunk_id=3)
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's "Founder Mode" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart.
This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale.
---
2) (chunk_id=1)
Paul Graham's essay "Fou

## 7) 再用两组测试问题对比


In [17]:
test_queries = [
    "what is the essay \"Founder Mode\" about?",
    "Why does conventional wisdom about giving autonomy fail for startups?",
]

for q in test_queries:
    print("=" * 80)
    print("QUERY:", q)

    print("\n[Propositions]")
    for i, r in enumerate(retriever_propositions.invoke(q), start=1):
        print(f"{i}) {r.page_content} (chunk_id={r.metadata['chunk_id']})")

    print("\n[Larger chunks]")
    for i, r in enumerate(retriever_larger.invoke(q), start=1):
        print(f"{i}) (chunk_id={r.metadata['chunk_id']})")
        print(r.page_content[:400].replace("\n", " ") + ("..." if len(r.page_content) > 400 else ""))

QUERY: what is the essay "Founder Mode" about?

[Propositions]
1) The essay "Founder Mode" challenges conventional wisdom about scaling startups. (chunk_id=1)
2) "Founder Mode" is an emerging paradigm that is not yet fully understood or documented. (chunk_id=1)
3) 'Founder Mode' is not yet fully understood or documented. (chunk_id=2)
4) Paul Graham published the essay "Founder Mode" in September 2024. (chunk_id=1)

[Larger chunks]
1) (chunk_id=1)
Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow. Conventional Wisdom vs. Founder Mode The essay argues that the traditional advice given to growing companies—hiring good people and gi...
2) (chunk_id=2)
Unique Founder Abilities Founders possess unique insights and abilities that professional managers do not, primarily because they